# Funding Eligibility Finder

## Notebook 1 — Funding Discovery Engine

### Project Motivation

Finding scholarships, grants, bursaries and funding opportunities is often a fragmented and inefficient process.

Relevant opportunities are distributed across university websites, foundations, charities, corporations, professional associations and government institutions. As a result, many potentially suitable funding opportunities remain undiscovered, particularly when eligibility criteria are highly specific.

This project aims to automate the discovery, collection and preliminary screening of funding opportunities through large-scale web search and profile-based filtering.

The broader objective is to build a complete Funding Eligibility Finder capable of identifying, extracting and ranking opportunities according to a student's characteristics and eligibility profile.

---

### Objective

This notebook identifies potentially relevant funding opportunities by:

- generating targeted funding-related search queries;
- performing automated web searches;
- collecting and deduplicating results;
- applying an initial relevance scoring framework;
- selecting candidates for deeper extraction in Notebook 2.

---

### Output

- `results/funding_search_raw_quick.csv`
- `results/funding_search_ranked_quick.csv`
- `results/funding_candidates_for_extraction.csv`

In [6]:
!pip install pandas ddgs tqdm openpyxl

In [7]:
# ============================================================
# FUNDING ELIGIBILITY FINDER
# Notebook 1: Funding Discovery Engine
# ============================================================

!pip install -q ddgs pandas openpyxl tqdm

import argparse
import logging
import random
import re
import time
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
from ddgs import DDGS
from ddgs.exceptions import DDGSException
from tqdm import tqdm


# ============================================================
# LOGGING
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

log = logging.getLogger("funding_discovery")


# ============================================================
# RUN MODES
# ============================================================

RUN_SETTINGS = {
    "quick": {
        "max_queries": 25,
        "max_results_per_query": 8,
        "pause_seconds": 1.5,
    },
    "standard": {
        "max_queries": 80,
        "max_results_per_query": 15,
        "pause_seconds": 1.8,
    },
    "deep": {
        "max_queries": 200,
        "max_results_per_query": 15,
        "pause_seconds": 2.2,
    },
}


# ============================================================
# APPLICANT PROFILE
# ============================================================

PROFILE = {
    "nationality": "Italian",
    "residence_country": "Italy",
    "gender": "female",
    "degree_level": "MSc",
    "destination_country": "UK",
    "start_year": 2026,
    "fields": [
        "financial mathematics",
        "quantitative finance",
        "mathematical finance",
        "data science",
        "finance",
        "STEM",
    ],
}


# ============================================================
# CURATED SEARCH QUERIES
# ============================================================

CURATED_QUERIES = [
    "scholarships for Italian students studying abroad 2026",
    "scholarships for women in STEM postgraduate UK 2026",
    "scholarships for women in finance master's students 2026",
    "grants for European women studying STEM in the UK",
    "corporate sponsorship for women studying finance UK",
    "foundation grants for international postgraduate students UK",
    "bursaries for international master's students UK 2026",
    "scholarships for quantitative finance master's students",
    "scholarships for women studying mathematics or data science",
    "private foundation scholarships for women in STEM Europe",
    "companies that sponsor women studying finance",
    "funding for Italian students pursuing a master's in the UK",
    "list of scholarships for international students UK universities 2026",
    "women in quantitative finance scholarship",
    "UK university scholarships for EU students after Brexit",
    "women in finance scholarship 2026",
    "women in finance scholarship master's students",
    "women in quantitative finance scholarship",
    "women in financial mathematics scholarship",
    "women in finance scholarship Europe 2026",
    "women in finance scholarship international students",
    "female students finance scholarship master",
    "female postgraduate finance scholarship",
    "central bank scholarship for women finance",
    "bank scholarship women finance students",
    "financial institution scholarship women finance",
    "corporate scholarship women in finance",
    "women in stem finance scholarship UK",
    "women in finance award scholarship 2026",
    "women in quantitative finance master's scholarship",
    "women in mathematical finance scholarship",
    "women in financial engineering scholarship",
    "women in fintech scholarship",
    "women in banking scholarship",
    "women in economics scholarship",
    "women in financial markets scholarship",
    "female finance students scholarship",
    "female quantitative finance students scholarship",
    "scholarships for female mathematicians",
    "women in applied mathematics scholarship",
    "master scholarship women economics finance",
]


# ============================================================
# COMBINATORIAL QUERY TERMS
# ============================================================

FUNDING_TERMS = [
    "scholarship",
    "grant",
    "bursary",
    "funding award",
]

APPLICANT_TERMS = [
    "women",
    "female students",
    "international students",
    "Italian students",
]

LOCATION_TERMS = [
    "UK",
    "London",
]


# ============================================================
# SCORING KEYWORDS
# ============================================================

FUNDING_KEYWORDS = [
    "scholarship",
    "scholarships",
    "grant",
    "grants",
    "bursary",
    "bursaries",
    "funding",
    "financial aid",
    "award",
    "awards",
    "sponsorship",
    "tuition support",
    "student support fund",
    "fellowship",
    "call for applications",
]

PROFILE_KEYWORDS = [
    "women",
    "female",
    "stem",
    "finance",
    "financial mathematics",
    "quantitative finance",
    "mathematical finance",
    "data science",
    "international students",
    "postgraduate",
    "msc",
    "master",
    "italian students",
    "european students",
    "uk",
    "united kingdom",
    "london",
    "study abroad",
]

NEGATIVE_KEYWORDS = [
    "phd only",
    "doctoral",
    "doctorate",
    "undergraduate only",
    "bachelor only",
    "bachelor's only",
    "uk citizens only",
    "british citizens only",
    "us citizens only",
    "u.s. citizens only",
    "usa residents only",
    "low income countries only",
    "low-income countries only",
    "developing countries only",
    "applications closed",
    "deadline passed",
    "closed for applications",
]

COURSE_PAGE_KEYWORDS = [
    "course overview",
    "programme structure",
    "program structure",
    "entry requirements",
    "curriculum",
    "modules",
    "apply for this course",
    "study this course",
]

AGGREGATOR_EXCLUSIONS = [
    "phdportal",
    "top scholarships",
    "scholarships by country",
    "study abroad scholarships",
    "scholarshiptab",
    "applyboard",
    "hotcourses",
    "opportunitydesk",
    "category/scholarships",
    "search our scholarships",
    "topuniversities",
    "masters degree scholarships",
    "facebook.com",
    "instagram.com",
    "linkedin.com",
    "youtube.com",
    "scholarshipsandgrants.us",
    "wemakescholars",
    "scholars4dev",
    "ukscholarships.uk",
    "gooverseas.com",
    "scholarshipsads.com",
    "scholarshipset.com",
    "scholarhunter.com",
    "findamasters.com/guides",
    "opportunitypool.org",
    "scholarshipsedge.com",
    "pathlins.com",
    "theeducationstory.com",
    "bheuni.io",
    "scholarshipsforwomen.net",
    "savethestudent.org",
    "pellopay.io",
    "womeninstemnetwork.com",
    "+ scholarships",
    "+ fully funded",
    "verified links",
    "fully funded masters",
    "ucas.com",
    "postgrad.com",
    "quantnet.com",
    "study-australia",
    "studyaustralia.gov.au",
    "tiktok.com",
    "reddit.com",
    "daadscholarship.com",
    "gmac.com",
    "bold.org",
    "ox.ac.uk/admissions/graduate/fees-and-funding",
    "imperial.ac.uk/",
    "advance-africa.com",
    "timeshighereducation.com",
    "topinternationalboardingschools",
    "careerlauncher.com",
    "european-funding-guide.eu",
    "nsf.gov",
]

AD_LINK_PATTERNS = [
    "bing.com/aclick",
    "googleadservices.com",
    "google.com/aclk",
    "doubleclick.net",
    "/aclick?",
    "msclkid=",
]

DEGREE_EXCLUSIONS = [
    "phd",
    "doctoral",
    "doctorate",
    "undergraduate",
    "bachelor",
]


# ============================================================
# DATA MODEL
# ============================================================

@dataclass
class RunConfig:
    mode: str
    max_queries: int
    max_results_per_query: int
    pause_seconds: float
    output_dir: Path
    max_retries: int = 3

    @property
    def raw_csv(self) -> Path:
        return self.output_dir / f"funding_search_raw_{self.mode}.csv"

    @property
    def ranked_csv(self) -> Path:
        return self.output_dir / f"funding_search_ranked_{self.mode}.csv"

    @property
    def ranked_xlsx(self) -> Path:
        return self.output_dir / f"funding_search_ranked_{self.mode}.xlsx"

    @property
    def candidates_csv(self) -> Path:
        return self.output_dir / "funding_candidates_for_extraction.csv"

    @property
    def candidates_xlsx(self) -> Path:
        return self.output_dir / "funding_candidates_for_extraction.xlsx"


# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def clean_text(text) -> str:
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def unique_preserve_order(items):
    return list(dict.fromkeys(items))


# ============================================================
# QUERY GENERATION
# ============================================================

def generate_combinatorial_queries(profile: dict) -> list:
    queries = []

    for funding in FUNDING_TERMS:
        for applicant in APPLICANT_TERMS:
            for field_name in profile["fields"]:
                for location in LOCATION_TERMS:
                    queries.append(
                        f"{funding} for {applicant} in {field_name} {location} {profile['start_year']}"
                    )

    return unique_preserve_order(queries)


def generate_queries(profile: dict, max_queries: int) -> list:
    curated = unique_preserve_order(CURATED_QUERIES)
    combinatorial = generate_combinatorial_queries(profile)

    rng = random.Random(42)
    rng.shuffle(combinatorial)

    all_queries = unique_preserve_order(curated + combinatorial)

    if max_queries < len(curated):
        log.warning(
            "max_queries (%d) is smaller than the curated query list (%d).",
            max_queries,
            len(curated),
        )

    return all_queries[:max_queries]


# ============================================================
# WEB SEARCH
# ============================================================

def search_one_query(
    ddgs: DDGS,
    query: str,
    max_results: int,
    max_retries: int,
) -> list:
    for attempt in range(1, max_retries + 1):
        try:
            return list(ddgs.text(query, max_results=max_results))

        except DDGSException as error:
            wait = min(2 ** attempt + random.uniform(0, 1), 30)

            log.warning(
                "Search failed for %r (attempt %d/%d): %s -- retrying in %.1fs",
                query,
                attempt,
                max_retries,
                error,
                wait,
            )

            time.sleep(wait)

        except Exception as error:
            log.error("Unexpected error for %r: %s", query, error)
            return []

    log.error("Giving up on query after %d attempts: %r", max_retries, query)

    return []


def is_ad_link(url: str) -> bool:
    url_lower = (url or "").lower()

    return any(pattern in url_lower for pattern in AD_LINK_PATTERNS)


def search_web(queries: list, config: RunConfig) -> pd.DataFrame:
    rows = []

    with DDGS() as ddgs:
        for query in tqdm(queries, desc="Searching funding opportunities"):
            results = search_one_query(
                ddgs=ddgs,
                query=query,
                max_results=config.max_results_per_query,
                max_retries=config.max_retries,
            )

            for item in results:
                url = item.get("href", "")

                if is_ad_link(url):
                    continue

                rows.append({
                    "query": query,
                    "title": item.get("title", ""),
                    "url": url,
                    "snippet": item.get("body", ""),
                })

            time.sleep(config.pause_seconds + random.uniform(0, 0.5))

    return pd.DataFrame(rows)


# ============================================================
# SCORING
# ============================================================

def score_result(row: pd.Series) -> pd.Series:
    text = clean_text(
        f"{row.get('title', '')} {row.get('snippet', '')} {row.get('url', '')}"
    )

    funding_matches = [
        keyword for keyword in FUNDING_KEYWORDS
        if keyword in text
    ]

    profile_matches = [
        keyword for keyword in PROFILE_KEYWORDS
        if keyword in text
    ]

    negative_matches = [
        keyword for keyword in NEGATIVE_KEYWORDS
        if keyword in text
    ]

    course_page_matches = [
        keyword for keyword in COURSE_PAGE_KEYWORDS
        if keyword in text
    ]

    is_aggregator = any(
        keyword in text
        for keyword in AGGREGATOR_EXCLUSIONS
    )

    funding_score = len(funding_matches) * 3
    profile_score = len(profile_matches)

    penalty_score = (
        len(course_page_matches) * 3
        + len(negative_matches) * 5
        + (4 if is_aggregator else 0)
    )

    relevance_score = funding_score + profile_score - penalty_score

    if negative_matches:
        status = "likely not eligible"
    elif is_aggregator:
        status = "aggregator / directory"
    elif not funding_matches:
        status = "likely course page / not funding"
    elif relevance_score >= 12:
        status = "high potential"
    elif relevance_score >= 7:
        status = "medium potential"
    elif relevance_score >= 4:
        status = "low potential"
    else:
        status = "unclear"

    return pd.Series({
        "status": status,
        "relevance_score": relevance_score,
        "is_aggregator": is_aggregator,
        "funding_matches": ", ".join(funding_matches),
        "profile_matches": ", ".join(profile_matches),
        "negative_flags": ", ".join(negative_matches),
        "course_page_flags": ", ".join(course_page_matches),
    })


def is_candidate_for_extraction(row: pd.Series) -> bool:
    text = clean_text(
        f"{row.get('title', '')} {row.get('url', '')} {row.get('snippet', '')}"
    )

    if row.get("is_aggregator"):
        return False

    if any(keyword in text for keyword in DEGREE_EXCLUSIONS):
        return False

    if row["status"] == "high potential":
        return True

    if row["status"] == "medium potential" and row["relevance_score"] >= 10:
        return True

    return False


# ============================================================
# PIPELINE
# ============================================================

def run_discovery_pipeline(profile: dict, config: RunConfig):
    log.info("Run mode: %s", config.mode)

    queries = generate_queries(profile, config.max_queries)

    log.info(
        "Selected %d queries. Curated queries are always prioritized.",
        len(queries),
    )

    log.info("Running web search...")

    raw_results = search_web(queries, config)

    log.info("Raw results: %d", len(raw_results))

    config.output_dir.mkdir(parents=True, exist_ok=True)

    raw_results.to_csv(config.raw_csv, index=False)

    if raw_results.empty:
        log.warning("No results found.")
        return raw_results, raw_results

    ranked_results = (
        raw_results
        .dropna(subset=["url"])
        .drop_duplicates(subset=["url"])
        .reset_index(drop=True)
    )

    log.info("Unique URLs: %d", len(ranked_results))
    log.info("Scoring results...")

    scoring = ranked_results.apply(score_result, axis=1)

    ranked_results = pd.concat([ranked_results, scoring], axis=1)

    status_order = {
        "high potential": 1,
        "medium potential": 2,
        "low potential": 3,
        "unclear": 4,
        "likely course page / not funding": 5,
        "aggregator / directory": 6,
        "likely not eligible": 7,
    }

    ranked_results["status_rank"] = ranked_results["status"].map(status_order)

    ranked_results = (
        ranked_results
        .sort_values(
            by=["status_rank", "relevance_score"],
            ascending=[True, False],
        )
        .drop(columns=["status_rank"])
        .reset_index(drop=True)
    )

    ranked_results["selected_for_extraction"] = ranked_results.apply(
        is_candidate_for_extraction,
        axis=1,
    )

    selected_candidates = ranked_results[
        ranked_results["selected_for_extraction"]
    ].copy()

    selected_candidates = (
    selected_candidates
    .sort_values(by="relevance_score", ascending=False)
    .head(60)
    .copy()
    )

    ranked_results.to_csv(config.ranked_csv, index=False)
    selected_candidates.to_csv(config.candidates_csv, index=False)

    try:
        ranked_results.to_excel(config.ranked_xlsx, index=False)
        selected_candidates.to_excel(config.candidates_xlsx, index=False)

    except ImportError:
        log.warning(
            "openpyxl not installed. Excel export skipped, CSV files saved."
        )

    log.info("Discovery completed.")
    log.info("Ranked results saved to: %s", config.ranked_csv)
    log.info("Candidates for extraction saved to: %s", config.candidates_csv)
    log.info("Selected candidates: %d", len(selected_candidates))

    return ranked_results, selected_candidates


# ============================================================
# RUN NOTEBOOK 1
# ============================================================

config = RunConfig(
    mode="standard",
    max_queries=RUN_SETTINGS["standard"]["max_queries"],
    max_results_per_query=RUN_SETTINGS["standard"]["max_results_per_query"],
    pause_seconds=RUN_SETTINGS["standard"]["pause_seconds"],
    output_dir=Path("results"),
)

funding_results, selected_candidates = run_discovery_pipeline(
    PROFILE,
    config,
)

print("\nGenerated files:")
print(f"• {config.raw_csv}")
print(f"• {config.ranked_csv}")
print(f"• {config.ranked_xlsx}")
print(f"• {config.candidates_csv}")
print(f"• {config.candidates_xlsx}")

print("\nStatus distribution:")
print(funding_results["status"].value_counts())

print(f"\nDataset shape: {funding_results.shape}")
print(f"Selected candidates: {selected_candidates.shape}")

display(
    selected_candidates[
        [
            "status",
            "relevance_score",
            "title",
            "url",
        ]
    ].head(50)
)


# ============================================================
# OPTIONAL DOWNLOAD FOR NOTEBOOK 2
# ============================================================

from google.colab import files

files.download(str(config.candidates_csv))

Searching funding opportunities: 100%|██████████| 80/80 [05:12<00:00,  3.91s/it]



Generated files:
• results/funding_search_raw_standard.csv
• results/funding_search_ranked_standard.csv
• results/funding_search_ranked_standard.xlsx
• results/funding_candidates_for_extraction.csv
• results/funding_candidates_for_extraction.xlsx

Status distribution:
status
medium potential                    205
aggregator / directory              182
high potential                       88
low potential                        84
likely course page / not funding     76
likely not eligible                   6
unclear                               4
Name: count, dtype: int64

Dataset shape: (645, 12)
Selected candidates: (60, 12)


,status,relevance_score,title,url
0,high potential,27,Top 2026 Scholarships for Women: College Fundi...,https://www.fastweb.com/college-scholarships/a...
3,high potential,25,Funding & Scholarships for Women in Mathematic...,https://www.mathunion.org/cwm/resources/fundin...
4,high potential,23,Spärck AI Scholarship | UCL Scholarships and f...,https://www.ucl.ac.uk/scholarships/sparck-ai-s...
5,high potential,22,AMS :: Fellowships & Scholarships - American M...,https://www.ams.org/grants-awards/ams-fellowsh...
6,high potential,20,Scholarships for Master's Studies in the Unite...,https://www.educations.com/articles-and-advice...
7,high potential,19,Women in STEM Scholarships for Studying Overse...,https://www.icanstudent.com/women-in-stem-scho...
8,high potential,19,University College London Mathematics Scholars...,https://mucuruzi.com/university-college-london...
9,high potential,18,UCL Master’s Funding Opportunities 2026/27: Bu...,https://opportunitiesforyouth.org/2026/02/17/u...
10,high potential,18,Master's degree funding | Postgraduate study |...,https://www.lboro.ac.uk/study/postgraduate/fee...
11,high potential,18,UCL Master's Bursary | UCL Scholarships and fu...,https://www.ucl.ac.uk/scholarships/ucl-masters...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>